#  Big Data con PySpark — Notebook 4
## Agregaciones, GroupBy y Window Functions

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
# Window → módulo para definir ventanas de datos (Window Functions)
# Permite calcular rankings, acumulados, diferencias entre filas

spark = (
    SparkSession.builder
    .appName("Vuelos_Agregaciones")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

df = spark.read.parquet("/content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet")
df.cache()
print(f"Filas: {df.count():,} | Columnas: {len(df.columns)}")

Filas: 446,399 | Columnas: 13


---
## 1. Funciones de agregación básicas — `agg()`

In [3]:
# ── Métricas globales del dataset ──
df.agg(
    F.count("vuelo_id").alias("total_vuelos"),       # número de filas
    F.sum("pasajeros").alias("total_pasajeros"),      # suma
    F.avg("tarifa_usd").alias("tarifa_promedio"),     # promedio
    F.min("tarifa_usd").alias("tarifa_minima"),       # mínimo
    F.max("tarifa_usd").alias("tarifa_maxima"),       # máximo
    F.stddev("retraso_min").alias("desv_retraso"),    # desviación estándar
    F.countDistinct("aerolinea").alias("n_aerolineas")# valores únicos
).show()

+------------+---------------+-----------------+-------------+-------------+-----------------+------------+
|total_vuelos|total_pasajeros|  tarifa_promedio|tarifa_minima|tarifa_maxima|     desv_retraso|n_aerolineas|
+------------+---------------+-----------------+-------------+-------------+-----------------+------------+
|      446399|       51144801|424.7738628670764|         50.0|        800.0|66.67533954033833|           6|
+------------+---------------+-----------------+-------------+-------------+-----------------+------------+



---
## 2. `groupBy()` + `agg()` — Agregación por grupos

Es el equivalente al `GROUP BY` de SQL o el `groupby().agg()` de pandas.

In [8]:
resumen_aerolinea = (
    df
    .withColumn("ingreso_total_usd", F.col("tarifa_usd") * F.col("pasajeros"))
    .withColumn("es_puntual", F.when(F.col("retraso_min") <= 0, 1).otherwise(0)) # Added this line to create the 'es_puntual' column
    .groupBy("aerolinea")                 # agrupa todas las filas por aerolinea
    .agg(
        F.count("vuelo_id").alias("vuelos"),
        F.sum("pasajeros").alias("pasajeros_totales"),
        F.round(F.sum("ingreso_total_usd"), 0).alias("ingreso_usd"),
        F.round(F.avg("retraso_min"), 1).alias("retraso_promedio"),
        F.round(F.avg("es_puntual") * 100, 1).alias("pct_puntualidad")
        # avg(es_puntual) donde es_puntual es 0 o 1 → da la proporción
        # × 100 → lo convierte a porcentaje
    )
    .orderBy(F.desc("ingreso_usd"))       # ordena por ingreso descendente
)
resumen_aerolinea.show()

+---------+------+-----------------+-------------+----------------+---------------+
|aerolinea|vuelos|pasajeros_totales|  ingreso_usd|retraso_promedio|pct_puntualidad|
+---------+------+-----------------+-------------+----------------+---------------+
|  Avianca|156005|         17886346|7.602218649E9|            35.8|           70.2|
|    LATAM|111695|         12797192| 5.43665566E9|            36.1|           70.1|
|    Wingo| 67004|          7662580|3.254252592E9|            35.7|           70.4|
|  EasyFly| 44590|          5119247|2.184271101E9|            35.8|           70.3|
|   Satena| 44562|          5095293|2.157765055E9|            35.4|           70.6|
|  JetBlue| 22543|          2584143| 1.09307398E9|            35.2|           70.7|
+---------+------+-----------------+-------------+----------------+---------------+



In [10]:
# ── Rutas más populares ──
rutas_top = (
    df
    .withColumn("ruta", F.concat(F.col("origen"), F.lit(" → "), F.col("destino"))) # Added this line to create the missing 'ruta' column
    .groupBy("ruta")                      # ruta = "BOG → MDE" (columna creada antes)
    .agg(
        F.count("*").alias("frecuencia"),
        F.round(F.avg("tarifa_usd"), 2).alias("tarifa_promedio"),
        F.round(F.avg("pasajeros"), 0).alias("pasajeros_prom")
    )
    .orderBy(F.desc("frecuencia"))
    .limit(10)                            # solo el top 10
    # .limit(n) → equivalente a SQL LIMIT; acción que recupera n filas
)
rutas_top.show(truncate=False)

+---------+----------+---------------+--------------+
|ruta     |frecuencia|tarifa_promedio|pasajeros_prom|
+---------+----------+---------------+--------------+
|LET → PEI|5112      |422.85         |115.0         |
|VVC → CTG|5107      |428.84         |115.0         |
|MDE → MTR|5090      |424.71         |114.0         |
|MDE → CTG|5083      |425.52         |115.0         |
|MTR → LET|5083      |423.89         |115.0         |
|MDE → VVC|5082      |421.52         |115.0         |
|CLO → MDE|5062      |427.46         |115.0         |
|PEI → LET|5061      |422.04         |114.0         |
|SMR → CTG|5058      |426.59         |114.0         |
|SMR → MDE|5056      |426.03         |115.0         |
+---------+----------+---------------+--------------+



In [11]:
# ── GroupBy por múltiples columnas ──
ocupacion_por_clase_aerolinea = (
    df
    .groupBy("aerolinea", "clase")        # agrupar por 2 columnas simultáneamente
    .agg(
        F.count("*").alias("vuelos"),
        F.round(F.avg("tarifa_usd"), 2).alias("tarifa_prom")
    )
    .orderBy("aerolinea", "clase")
)
ocupacion_por_clase_aerolinea.show(20)

+---------+---------+------+-----------+
|aerolinea|    clase|vuelos|tarifa_prom|
+---------+---------+------+-----------+
|  Avianca| Business| 30991|     424.59|
|  Avianca|Economica|117270|     425.03|
|  Avianca|  Primera|  7744|      425.0|
|  EasyFly| Business|  8937|     428.35|
|  EasyFly|Economica| 33389|     426.44|
|  EasyFly|  Primera|  2264|     423.24|
|  JetBlue| Business|  4463|     421.54|
|  JetBlue|Economica| 17004|     424.93|
|  JetBlue|  Primera|  1076|     407.63|
|    LATAM| Business| 22286|     425.04|
|    LATAM|Economica| 83756|     424.59|
|    LATAM|  Primera|  5653|     421.83|
|   Satena| Business|  8794|     424.33|
|   Satena|Economica| 33577|      424.0|
|   Satena|  Primera|  2191|      421.9|
|    Wingo| Business| 13433|     426.73|
|    Wingo|Economica| 50371|     423.76|
|    Wingo|  Primera|  3200|     426.93|
+---------+---------+------+-----------+



In [13]:
# ── Pivot: aerolinea × mes → total de vuelos ──
pivot_mensual = (
    df
    .groupBy("aerolinea")                 # filas del pivot
    .pivot("mes", list(range(1, 13)))     # columnas del pivot: meses 1 a 12
    # pivot(col, [valores]) → crea una columna por cada valor único de 'col'
    .agg(F.count(F.lit(1)))              # Changed from F.count("*") to F.count(F.lit(1))
    .orderBy("aerolinea")
)
pivot_mensual.show()

+---------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|aerolinea|    1|    2|    3|    4|    5|    6|    7|    8|    9|   10|   11|   12|
+---------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|  Avianca|13274|11940|13151|13004|13117|12842|13359|13457|12831|13223|12799|13008|
|  EasyFly| 3822| 3412| 3908| 3687| 3824| 3592| 3746| 3781| 3684| 3878| 3684| 3572|
|  JetBlue| 1947| 1698| 1934| 1815| 1806| 1836| 1894| 1965| 1815| 1922| 1942| 1969|
|    LATAM| 9376| 8475| 9400| 9213| 9404| 9129| 9609| 9622| 9211| 9496| 9230| 9530|
|   Satena| 3762| 3479| 3883| 3547| 3906| 3634| 3740| 3724| 3679| 3780| 3644| 3784|
|    Wingo| 5644| 5143| 5643| 5498| 5835| 5636| 5735| 5669| 5511| 5688| 5407| 5595|
+---------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+



---
## 3. Window Functions — Funciones de Ventana

Las Window Functions calculan un valor para cada fila **usando información de un grupo de filas relacionadas**, sin reducir el número de filas como hace `groupBy`.

```
groupBy + agg  →  1 fila por grupo  (reduce)
Window func    →  mantiene todas las filas, agrega columna con el cálculo de ventana
```

In [14]:
# ── Definir una ventana ──
ventana_aerolinea = (
    Window
    .partitionBy("aerolinea")    # equivalente a GROUP BY — define el grupo
    .orderBy(F.desc("tarifa_usd"))  # define el orden dentro de cada grupo
)
# Window es el objeto que define "sobre qué conjunto de filas" opera la función
# partitionBy → agrupa (como GROUP BY)
# orderBy     → ordena dentro del grupo (necesario para rank, lead, lag, etc.)

In [16]:
# ── Ranking dentro de grupo: rank() ──
df_with_ruta = df.withColumn("ruta", F.concat(F.col("origen"), F.lit(" → "), F.col("destino"))) # Added this line to create the 'ruta' column

df_ranking = df_with_ruta.withColumn(
    "rank_tarifa",
    F.rank().over(ventana_aerolinea)
    # rank() → asigna el puesto dentro de la ventana (1 = mayor tarifa)
    # .over(ventana) → especifica sobre qué ventana opera
    # rank() deja huecos si hay empates (1, 2, 2, 4)
    # dense_rank() no deja huecos (1, 2, 2, 3)
    # row_number() numeración única sin empates (1, 2, 3, 4)
)

# Ver el vuelo más caro por aerolinea
(
    df_ranking
    .filter(F.col("rank_tarifa") == 1)    # solo el puesto #1 de cada grupo
    .select("aerolinea", "ruta", "tarifa_usd", "rank_tarifa")
    .orderBy("aerolinea")
    .show()
)

+---------+---------+----------+-----------+
|aerolinea|     ruta|tarifa_usd|rank_tarifa|
+---------+---------+----------+-----------+
|  Avianca|BAQ → MTR|     800.0|          1|
|  EasyFly|CLO → MDE|    799.99|          1|
|  EasyFly|SMR → MTR|    799.99|          1|
|  JetBlue|MDE → VVC|    799.96|          1|
|    LATAM|MDE → SMR|    799.99|          1|
|    LATAM|PEI → VVC|    799.99|          1|
|    LATAM|SMR → BAQ|    799.99|          1|
|    LATAM|BAQ → CLO|    799.99|          1|
|   Satena|MTR → CLO|    799.96|          1|
|    Wingo|MDE → PEI|     800.0|          1|
|    Wingo|MTR → BOG|     800.0|          1|
+---------+---------+----------+-----------+



In [17]:
# ── Comparar cada vuelo con el promedio de su aerolinea ──
ventana_sin_orden = Window.partitionBy("aerolinea")
# Para avg, sum, min, max no se necesita orderBy (no dependen del orden)

df_vs_promedio = df.withColumn(
    "promedio_aerolinea",
    F.round(F.avg("tarifa_usd").over(ventana_sin_orden), 2)
    # avg(col).over(ventana) → calcula el promedio del grupo
    # pero lo añade a CADA FILA del grupo (no reduce)
).withColumn(
    "diff_vs_promedio",
    F.round(F.col("tarifa_usd") - F.col("promedio_aerolinea"), 2)
    # cuánto se desvía este vuelo del promedio de su aerolinea
)

df_vs_promedio.select(
    "aerolinea", "tarifa_usd", "promedio_aerolinea", "diff_vs_promedio"
).orderBy("aerolinea", F.desc("diff_vs_promedio")).show(10)

+---------+----------+------------------+----------------+
|aerolinea|tarifa_usd|promedio_aerolinea|diff_vs_promedio|
+---------+----------+------------------+----------------+
|  Avianca|     800.0|            424.94|          375.06|
|  Avianca|    799.99|            424.94|          375.05|
|  Avianca|    799.98|            424.94|          375.04|
|  Avianca|    799.98|            424.94|          375.04|
|  Avianca|    799.98|            424.94|          375.04|
|  Avianca|    799.97|            424.94|          375.03|
|  Avianca|    799.97|            424.94|          375.03|
|  Avianca|    799.97|            424.94|          375.03|
|  Avianca|    799.95|            424.94|          375.01|
|  Avianca|    799.94|            424.94|           375.0|
+---------+----------+------------------+----------------+
only showing top 10 rows


In [18]:
# ── Acumulados (running total) por mes ──
ventana_mes_acum = (
    Window
    .partitionBy("aerolinea")
    .orderBy("mes")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    # rowsBetween(inicio, fin):
    #   Window.unboundedPreceding → desde el inicio del grupo
    #   Window.currentRow         → hasta la fila actual
    # Esto define una ventana que crece: incluye todas las filas anteriores + la actual
)

vuelos_por_mes = (
    df.groupBy("aerolinea", "mes")
    .agg(F.count("*").alias("vuelos_mes"))
)

vuelos_acumulados = vuelos_por_mes.withColumn(
    "vuelos_acumulados",
    F.sum("vuelos_mes").over(ventana_mes_acum)
)

(
    vuelos_acumulados
    .filter(F.col("aerolinea") == "Avianca")
    .orderBy("mes")
    .show()
)

+---------+---+----------+-----------------+
|aerolinea|mes|vuelos_mes|vuelos_acumulados|
+---------+---+----------+-----------------+
|  Avianca|  1|     13274|            13274|
|  Avianca|  2|     11940|            25214|
|  Avianca|  3|     13151|            38365|
|  Avianca|  4|     13004|            51369|
|  Avianca|  5|     13117|            64486|
|  Avianca|  6|     12842|            77328|
|  Avianca|  7|     13359|            90687|
|  Avianca|  8|     13457|           104144|
|  Avianca|  9|     12831|           116975|
|  Avianca| 10|     13223|           130198|
|  Avianca| 11|     12799|           142997|
|  Avianca| 12|     13008|           156005|
+---------+---+----------+-----------------+



In [19]:
# ── lag() / lead(): acceder a filas anteriores/siguientes ──
ventana_mes_ord = Window.partitionBy("aerolinea").orderBy("mes")

evolucion_mensual = (
    df.groupBy("aerolinea", "mes")
    .agg(F.round(F.avg("tarifa_usd"), 2).alias("tarifa_prom_mes"))
    .withColumn(
        "tarifa_mes_anterior",
        F.lag("tarifa_prom_mes", 1).over(ventana_mes_ord)
        # lag(col, n) → devuelve el valor de n filas ANTES dentro de la ventana
        # Si es la primera fila del grupo → devuelve null
    )
    .withColumn(
        "variacion_pct",
        F.round(
            (F.col("tarifa_prom_mes") - F.col("tarifa_mes_anterior")) /
            F.col("tarifa_mes_anterior") * 100, 1
        )
    )
    # lead(col, n) → valor de n filas DESPUÉS (para forecasting)
)

evolucion_mensual.filter(F.col("aerolinea") == "Avianca").orderBy("mes").show()

+---------+---+---------------+-------------------+-------------+
|aerolinea|mes|tarifa_prom_mes|tarifa_mes_anterior|variacion_pct|
+---------+---+---------------+-------------------+-------------+
|  Avianca|  1|         425.37|               NULL|         NULL|
|  Avianca|  2|         424.17|             425.37|         -0.3|
|  Avianca|  3|         425.71|             424.17|          0.4|
|  Avianca|  4|         424.69|             425.71|         -0.2|
|  Avianca|  5|         427.81|             424.69|          0.7|
|  Avianca|  6|          424.9|             427.81|         -0.7|
|  Avianca|  7|         422.55|              424.9|         -0.6|
|  Avianca|  8|         427.14|             422.55|          1.1|
|  Avianca|  9|         424.02|             427.14|         -0.7|
|  Avianca| 10|         424.24|             424.02|          0.1|
|  Avianca| 11|         423.74|             424.24|         -0.1|
|  Avianca| 12|         424.76|             423.74|          0.2|
+---------

---
## Resumen del notebook

```
df.agg(F.count(), F.sum(), F.avg(), ...)      →  métricas globales
df.groupBy(col).agg(...)                      →  métricas por grupo
df.groupBy(col).pivot(col2).agg(...)          →  tabla dinámica

Window.partitionBy(col).orderBy(col)          →  definir ventana
F.rank().over(ventana)                        →  ranking dentro del grupo
F.avg(col).over(ventana)                      →  promedio por grupo (sin reducir filas)
F.sum(col).over(ventana_acum)                 →  acumulado
F.lag(col, n).over(ventana)                   →  valor n filas antes
F.lead(col, n).over(ventana)                  →  valor n filas después
```

 **Siguiente:** Spark SQL — consultas SQL sobre DataFrames